In [ ]:
ingest_date = None
data_format = None   # "json" or "xml"

In [ ]:
%run 00_config

In [ ]:
import os, re, uuid, requests
from datetime import datetime, timezone, date
from pyspark.sql import Row

run_id = uuid.uuid4().hex
ingest_date = ingest_date or date.today().isoformat()
fmt = (data_format or fmt).lower()
accept = f"application/fhir+{fmt}"

In [ ]:
def read_watermark(resource):
    try:
        rows = (spark.read.table("control_watermark")
                .where(f"resource = '{resource}' AND format = '{fmt}'")
                .orderBy("updated_at", ascending=False).limit(1).collect())
        return rows[0]["last_updated"] if rows else seed_date
    except Exception:
        return seed_date

def advance_watermark(resource, value):
    row = Row(resource=resource, format=fmt, last_updated=value,
              updated_at=datetime.now(timezone.utc).isoformat())
    (spark.createDataFrame([row]).write.format("delta")
     .mode("append").option("mergeSchema", "true").saveAsTable("control_watermark"))

In [ ]:
def fetch_pages(resource, updated_after):
    s = requests.Session()
    s.headers.update({"Accept": accept})
    url = f"{base_url}/{resource}"
    params = {"_count": page_size, "_sort": "_lastUpdated", "_lastUpdated": f"ge{updated_after}"}
    for page in range(1, max_pages + 1):
        r = s.get(url, params=params, timeout=60)
        r.raise_for_status()
        text = r.text
        if fmt == "json":
            bundle = r.json()
            entries = bundle.get("entry", []) or []
            count = len(entries)
            last_updates = [e.get("resource", {}).get("meta", {}).get("lastUpdated") for e in entries]
            nxt = next((l["url"] for l in bundle.get("link", []) if l["relation"] == "next"), None)
        else:
            count = len(re.findall(r"<entry[ >]", text))
            last_updates = re.findall(r'<lastUpdated value="([^"]+)"', text)
            nxt = next((re.search(r'<url value="([^"]+)"', b).group(1)
                        for b in re.findall(r"<link>.*?</link>", text, re.DOTALL)
                        if 'value="next"' in b), None)
        yield page, r.url, text, count, last_updates
        if not nxt or count == 0:
            break
        url, params = nxt, None

In [ ]:
audit = []
for resource in resources:
    wm = read_watermark(resource)
    out_dir = raw_dir(resource, ingest_date, fmt)
    os.makedirs(out_dir, exist_ok=True)
    newest, total = wm, 0
    for page, req_url, body, count, last_updates in fetch_pages(resource, wm):
        fetched = datetime.now(timezone.utc).isoformat()
        with open(f"{out_dir}/{resource}_page{page:03d}.{fmt}", "w", encoding="utf-8") as f:
            f.write(body)
        saved = datetime.now(timezone.utc).isoformat()
        total += count
        newest = max([newest] + [lu for lu in last_updates if lu])
        audit.append((run_id, resource, fmt, page, req_url, fetched, saved, count, ingest_date))
        print(f"[{fmt}] {resource} p{page}: {count}")
    advance_watermark(resource, newest)
    print(f"  {resource}: {total} records, watermark {newest}")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema = StructType([
    StructField("run_id", StringType()), StructField("resource", StringType()),
    StructField("format", StringType()), StructField("page", IntegerType()),
    StructField("api_url_or_params", StringType()), StructField("extraction_timestamp", StringType()),
    StructField("load_timestamp", StringType()), StructField("record_count", IntegerType()),
    StructField("ingest_date", StringType())])
(spark.createDataFrame(audit, schema).write.format("delta")
 .mode("append").option("mergeSchema", "true").saveAsTable("control_audit"))
display(spark.read.table("control_audit").orderBy("extraction_timestamp", ascending=False))